# Sensor tasking ↔ portfolio optimization

Side-by-side run to convince myself the sensor tasking formulation
really is the same QP I've been writing for portfolio allocation.

Step 1 - portfolio: max μᵀw - γ wᵀΣw  s.t.  1ᵀw = 1, w ≥ 0, w ≤ caps.
Step 2 - tasking:  max vᵀd - γ d̃ᵀΣd̃  s.t. Σdᵢ ≤ B, dᵢ ≥ dmin, dᵢ ≤ dmax,
                                            per-priority caps.

The only structural difference is the budget constraint becoming an
inequality (you don't have to spend all the dwell time) and a
rebranding of the per-asset / per-target caps.

In [ ]:
import sys, numpy as np
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'quant' / 'src'))

from geoalpha_quant.optimization import SensorTaskingProblem, TargetRequest, solve_sensor_tasking

rng = np.random.default_rng(0)
n = 12
targets = [
    TargetRequest(
        name=f't{i:02d}',
        value=float(rng.uniform(2.0, 12.0)),
        dwell_max=30.0,
        priority=rng.choice(['ROUTINE', 'PRIORITY', 'IMMEDIATE', 'FLASH']),
    )
    for i in range(n)
]
for t in targets:
    print(f'{t.name}  v={t.value:5.2f}  prio={t.priority}')

In [ ]:
problem = SensorTaskingProblem(targets=targets, total_budget_s=120.0, risk_aversion=0.5)
result = solve_sensor_tasking(problem)
print('solver:', result.solver)
print('total value:', round(result.total_value, 2))
for k, v in sorted(result.as_assignment(problem).items(), key=lambda kv: -kv[1]):
    print(f'  {k}  dwell={v:5.1f}s')

Greedy + CVXPY both honour the per-priority caps - FLASH gets fully
loaded first, then IMMEDIATE, etc.  This was the headline I wanted to
verify before the dashboard's tasking page ships.
